In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
from collections import defaultdict
from PIL import Image
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report

import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, GlobalAveragePooling2D, Dense, 
    BatchNormalization, Activation, Dropout
)
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2

SEED = 4242
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

img_dir = '../flipped_images'
excel_path = '../tabular.xlsx'

variant_keys = ['west_original', 'east_flip_horizontal', 'north_flip_vertical', 'south_flip_both']
image_names_to_select = ['AC QPS (clear)', 'Rest QGS (clear)', 'Stress QGS (clear)']
target_labels = ['normal', 'ischemia', 'infarction']

def pad_image(img, target_size=(224, 224)):
    width, height = img.size
    new_width, new_height = target_size
    if width > new_width or height > new_height:
        img.thumbnail((new_width, new_height), Image.LANCZOS)
        width, height = img.size 
    padding_width = (new_width - width) // 2
    padding_height = (new_height - height) // 2
    return np.pad(np.array(img),
                  ((padding_height, new_height - height - padding_height),
                   (padding_width, new_width - width - padding_width),
                   (0, 0)),
                  mode='constant', constant_values=0)

def plot_and_save_history(history, title, filename):
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(filename)
    plt.close()
    print(f"Saved {title} training plot to {filename}")

data = pd.read_excel(excel_path)
label_map = {'Normal': 0, 'Ischemia': 1, 'Infarction': 2}
data['Label'] = data['Label'].map(label_map)
data = pd.get_dummies(data, columns=['Sex'])

features = data.drop(['Label', 'ID'], axis=1)
correlation = features.corr().abs()
upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] == 1)]
data = data.drop(columns=to_drop)

tab_cols = [c for c in data.columns if c not in ['ID', 'Label']]
patient_data_dict = {}
for _, row in data.iterrows():
    patient_data_dict[int(row['ID'])] = {
        'features': row[tab_cols].values.astype(float),
        'label': int(row['Label'])
    }

X_img, X_tab, y_matched = [], [], []

for folder in os.listdir(img_dir):
    folder_path = os.path.join(img_dir, folder)
    if not os.path.isdir(folder_path): continue

    label_idx = next((target_labels.index(lbl) for lbl in target_labels if lbl in folder.lower()), None)
    if label_idx is None: continue

    files = [f for f in os.listdir(folder_path) if f.startswith('cropped')]
    grouped = defaultdict(lambda: defaultdict(dict))
    
    for f in files:
        match = re.match(r'cropped_(\d+)\s+(.+?)_(.+?)\.jpg$', f) 
        if match:
            pat_id, view_name, variant_key = match.groups()
            pat_id = int(pat_id)
            if view_name in image_names_to_select and variant_key in variant_keys:
                grouped[pat_id][variant_key][view_name] = f

    for pat_id, variants in grouped.items():
        if pat_id not in patient_data_dict: continue
            
        pat_tabular = patient_data_dict[pat_id]['features']
        pat_label = patient_data_dict[pat_id]['label']
        
        for variant_key in variant_keys:
            views = variants.get(variant_key, {})
            patient_images = []
            
            for view in image_names_to_select:
                matched_file = views.get(view)
                if matched_file:
                    img_path = os.path.join(folder_path, matched_file)
                    img = Image.open(img_path).convert('RGB')
                    padded_img = pad_image(img, target_size=(224, 224))
                    img_arr = preprocess_input(np.array(padded_img))
                else:
                    img_arr = preprocess_input(np.zeros((224, 224, 3), dtype=np.float32))
                patient_images.append(img_arr)
                
            stacked = np.concatenate(patient_images, axis=-1)
            X_img.append(stacked)
            X_tab.append(pat_tabular)
            y_matched.append(pat_label)

X_img = np.array(X_img)
X_tab = np.array(X_tab)
y_cat = to_categorical(np.array(y_matched), num_classes=3)

print(f"Data Aligned. Total combined samples: {X_img.shape[0]}")


X_train_img, X_val_img, X_train_tab, X_test_tab, y_train_cat, y_test_cat = train_test_split(
    X_img, X_tab, y_cat, test_size=0.2, random_state=42, stratify=np.argmax(y_cat, axis=1)
)

raw_mean_vals = np.nanmean(X_train_tab, axis=0)
X_train_tab = np.nan_to_num(X_train_tab, nan=raw_mean_vals)
X_test_tab = np.nan_to_num(X_test_tab, nan=raw_mean_vals) 

# ANOVA
selector = SelectKBest(score_func=f_classif, k=20)
X_train_sel = selector.fit_transform(X_train_tab, np.argmax(y_train_cat, axis=1))
X_test_sel = selector.transform(X_test_tab)


selected_mask = selector.get_support()
selected_features = np.array(tab_cols)[selected_mask].tolist() # Extract the exact 20 feature names chosen by ANOVA
print("\nTop 20 Features selected by ANOVA:")
print(selected_features)

scaler = MinMaxScaler()
X_train_sel = scaler.fit_transform(X_train_sel)
X_test_sel = scaler.transform(X_test_sel)

# the scaled means of just the 20 features to save for the UI
scaled_selected_means = np.nanmean(X_train_sel, axis=0)

input_9ch = Input(shape=(224, 224, 9), name="input_9channel")
reduced = Conv2D(3, (1, 1), activation='relu')(input_9ch)
resnet_base = ResNet50(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
resnet_base.trainable = False 
for layer in resnet_base.layers[-10:]: layer.trainable = True

x = resnet_base(reduced)
x = GlobalAveragePooling2D()(x)
x = Dense(64, kernel_regularizer=l2(0.005))(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.7)(x)
output = Dense(3, activation='softmax', kernel_regularizer=l2(0.005))(x)

cnn_model = Model(inputs=input_9ch, outputs=output)
cnn_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002), 
                  loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.005), 
                  metrics=['accuracy'])

cnn_history = cnn_model.fit(X_train_img, y_train_cat, validation_data=(X_val_img, y_test_cat), 
              epochs=50, batch_size=32, callbacks=[ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)])

plot_and_save_history(cnn_history, "CNN Model", "cnn_training_history.png")

y_train_labels_flat = np.argmax(y_train_cat, axis=1)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_labels_flat), y=y_train_labels_flat)
class_weight_dict = {0: class_weights[0], 1: class_weights[1], 2: class_weights[2]}

ann_model = Sequential([
    Input(shape=(20,)),
    Dense(64, activation='relu'),            
    Dense(64, activation='relu'),             
    Dense(3, activation='softmax')           
])
ann_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.002), loss='categorical_crossentropy', metrics=['accuracy'])

ann_history = ann_model.fit(X_train_sel, y_train_cat, epochs=65, batch_size=32, validation_data=(X_test_sel, y_test_cat), 
              callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)], class_weight=class_weight_dict)

plot_and_save_history(ann_history, "ANN Model", "ann_training_history.png")

cnn_pred = cnn_model.predict(X_val_img)
ann_pred = ann_model.predict(X_test_sel)

fused_pred = (0.6 * cnn_pred) + (0.4 * ann_pred)
final_classes = np.argmax(fused_pred, axis=1)
true_classes = np.argmax(y_test_cat, axis=1)

print(f"\nFused Model Accuracy: {accuracy_score(true_classes, final_classes):.4f}")
print(classification_report(true_classes, final_classes, target_names=target_labels))

os.makedirs("production_models", exist_ok=True)
cnn_model.save("production_models/cnn_model.keras")
ann_model.save("production_models/ann_model.keras")

joblib.dump({
    'scaler': scaler,
    'mean_vals': scaled_selected_means, # Updated to only hold the 20 selected feature means
    'selected_columns': selected_features 
}, "production_models/tabular_pipeline.pkl")

Data Aligned. Total combined samples: 388

Top 20 Features selected by ANOVA:
['SSS', 'SRS', 'SDS', 'SS%', 'SR%', 'SD%', 'Stress_QPS_Defect', 'Stress_QPS_Extent', 'Stress_QPS_TPD', 'Rest_QPS_Defect', 'Rest_QPS_Extent', 'Rest_QPS_TPD', 'Stress_QGS_SMS', 'Stress_QGS_STS', 'Stress_QGS_SM%', 'Stress_QGS_ST%', 'Stress_QGS_Mot_Ext', 'Stress_QGS_Thk_Ext', 'Rest_QGS_SMS', 'Rest_QGS_Thk_Ext']
Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 35s 2s/step - accuracy: 0.3548 - loss: 2.2957 - val_accuracy: 0.4872 - val_loss: 1.6741 - learning_rate: 2.0000e-04
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - accuracy: 0.4581 - loss: 1.9288 - val_accuracy: 0.6282 - val_loss: 1.4967 - learning_rate: 2.0000e-04
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 17s 2s/step - accuracy: 0.6032 - loss: 1.5626 - val_accuracy: 0.6538 - val_loss: 1.4234 - learning_rate: 2.0000e-04
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - accuracy: 0.7097 - loss: 1.4004 - val_accuracy: 0.7308 - val_loss: 1.3316 - learning_rate: 2.0000e-

['production_models/tabular_pipeline.pkl']

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import joblib
import gradio as gr

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input

CNN_MODEL_PATH = "production_models/cnn_model.keras"
ANN_MODEL_PATH = "production_models/ann_model.keras"
PIPELINE_PATH = "production_models/tabular_pipeline.pkl"

cnn_model = load_model(CNN_MODEL_PATH)
ann_model = load_model(ANN_MODEL_PATH)
pipeline = joblib.load(PIPELINE_PATH)

scaler = pipeline['scaler']
mean_vals = pipeline['mean_vals']
expected_columns = pipeline['selected_columns'] 

demo_columns = expected_columns[:5]

CROP_BOX = (320, 0, 950, 930) 

def crop_and_pad(img, target_size=(224, 224)):
    img = img.crop(CROP_BOX)
    width, height = img.size
    new_width, new_height = target_size

    if width > new_width or height > new_height:
        img.thumbnail((new_width, new_height), Image.LANCZOS)
        width, height = img.size

    padding_width = (new_width - width) // 2
    padding_height = (new_height - height) // 2

    padded_img = np.pad(np.array(img),
                        ((padding_height, new_height - height - padding_height),
                         (padding_width, new_width - width - padding_width),
                         (0, 0)),
                        mode='constant', constant_values=0)
    return padded_img

def preprocess_patient_images(img_ac_qps, img_rest_qgs, img_stress_qgs):
    images = [img_ac_qps, img_rest_qgs, img_stress_qgs]
    processed_arrays = []
    
    for img in images:
        if img is not None:
            img = img.convert('RGB')
            padded = crop_and_pad(img, target_size=(224, 224))
            arr = preprocess_input(np.array(padded, dtype=np.float32))
        else:
            arr = preprocess_input(np.zeros((224, 224, 3), dtype=np.float32))
        processed_arrays.append(arr)
        
    stacked = np.concatenate(processed_arrays, axis=-1)
    return np.expand_dims(stacked, axis=0) # Batch size 1

def predict_patient(img_ac, img_rest, img_stress, *tabular_values):
    labels = ['Normal', 'Ischemia', 'Infarction']
    
    if not all([img_ac, img_rest, img_stress]):
        return {"Error: Please upload all three required SPECT images.": 1.0}
    
    X_img = preprocess_patient_images(img_ac, img_rest, img_stress)
    
    
    df_dict = {col: [val] for col, val in zip(demo_columns, tabular_values)}# map the 5 UI inputs to their respective column names
    df = pd.DataFrame(df_dict)
    

    for col in expected_columns:   # pad the remaining 15 hidden features with NaN
        if col not in df.columns:
            df[col] = np.nan
            
    df = df[expected_columns]
    

    X_scaled = scaler.transform(df.values)
    X_scaled = np.nan_to_num(X_scaled, nan=mean_vals) # scaler ignores NaNs and nan_to_num replaces them with the trained averages
    
    cnn_pred = cnn_model.predict(X_img)
    ann_pred = ann_model.predict(X_scaled)
    
    fused_pred = (0.6 * cnn_pred) + (0.4 * ann_pred)
    fusion_probabilities = fused_pred[0]
    
    result = {labels[i]: float(fusion_probabilities[i]) for i in range(len(labels))}
    return result

with gr.Blocks(title="MPI Classification Model") as demo:
    gr.Markdown("# Multimodal Diagnostic Inference")
    gr.Markdown("Upload the 3 required SPECT scans and input the top 5 key clinical features to generate a fused diagnosis.")
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1. Image Data")
            img_ac = gr.Image(type="pil", label="AC QPS (Clear)")
            img_rest = gr.Image(type="pil", label="Rest QGS (Clear)")
            img_stress = gr.Image(type="pil", label="Stress QGS (Clear)")
            
        with gr.Column(scale=1):
            gr.Markdown("### 2. Tabular Data")
            
            tabular_inputs = []
            with gr.Accordion("Patient Clinical Metrics", open=True):
                # Iterate ONLY over the top 5 columns
                for col_name in demo_columns:
                    tab_input = gr.Number(label=col_name, value=0.0)
                    tabular_inputs.append(tab_input)
            
            submit_btn = gr.Button("Analyze Patient Data", variant="primary")
            
            gr.Markdown("### 3. Diagnosis Output")
            output_label = gr.Label(num_top_classes=3)

    submit_btn.click(
        fn=predict_patient,
        inputs=[img_ac, img_rest, img_stress] + tabular_inputs, 
        outputs=[output_label]
    )

if __name__ == "__main__":
    demo.launch(share=True)